# Python Dictionaries — practice notebook

**Kaise use karein:** har cell ka output pehle socho, phir Run dabao. Do cell jaan-boojh kar error dete hain — unka error padho, phir fix wale cell chalao.

Sirf dictionaries: lookup, iteration, counting, grouping, merging, nesting aur debugging.


## 1. Creation aur safe lookup

`d[key]` missing key par **KeyError** deta hai (fail fast). `d.get(key, default)` missing par default deta hai. Insertion order 3.7+ me guaranteed hai.


In [ ]:
student = {'name': 'Neha', 'marks': 88, 'course': 'AI'}
print(student['name'], student.get('city', 'unknown'))

student['marks'] = 92        # update
student['city'] = 'Pune'     # naya key
print(student)

print(list(student), list(student.values()))
print('city' in student, 'phone' in student)

pairs = list(student.items())
print(pairs[0], dict(pairs)['name'])

print(dict.fromkeys(['a', 'b'], 0))

## 2. Delete, merge aur precedence

`pop(key, default)` safe delete deta hai. Merge me **right side ki value jeet ti hai** — `d1 | d2` nayi dict deta hai, `update` in-place karta hai. Dono shallow hain.


In [ ]:
d = {'a': 1, 'b': 2, 'c': 3}
print(d.pop('a', None), d.pop('zzz', 'missing'))
print(d)

defaults = {'timeout': 30, 'retries': 2}
overrides = {'timeout': 60}
print('|       :', defaults | overrides)
print('unpacked:', {**defaults, **overrides})

in_place = dict(defaults)
in_place.update(overrides)
print('update  :', in_place)

nested = {'db': {'host': 'a', 'port': 1}, 'cache': {'ttl': 60}}
print('shallow merge:', {**nested, 'db': {'port': 2}})

## 3. Counting aur frequency (Counter)

`Counter` missing key par `0` deta hai — isliye counting one-liner ban jati hai. Manual loop bhi sikho, kyunki interview me tumse likhwate hain.


In [ ]:
from collections import Counter

text = 'data is data science and data is powerful'
print(Counter(text.split()).most_common(3))

# manual version (interview me yahi likhna padta hai)
freq = {}
for word in text.split():
    freq[word] = freq.get(word, 0) + 1
print(sorted(freq.items(), key=lambda kv: -kv[1])[:3])

chars = {}
for ch in 'copy function getting failed':
    chars[ch] = chars.get(ch, 0) + 1
print(dict(sorted(chars.items(), key=lambda kv: -kv[1])[:5]))

## 4. Grouping (defaultdict) aur sorting

`defaultdict(list)` me naya key automatically khali list deta hai — `if key not in d` boilerplate khatam. Sorting value par `key=lambda kv: kv[1]` se hoti hai.


In [ ]:
from collections import defaultdict

rows = [('nlp', 120), ('cv', 80), ('nlp', 90), ('cv', 100)]
groups = defaultdict(list)
for cat, val in rows:
    groups[cat].append(val)
print(dict(groups))

summary = {cat: {'n': len(v), 'total': sum(v), 'avg': round(sum(v) / len(v), 1)} for cat, v in groups.items()}
print(summary)

prices = {'OnePlus': 20000, 'Redmi': 12000, 'Poco': 11500, 'Samsung': 14500}
print('by value :', dict(sorted(prices.items(), key=lambda kv: kv[1])))
print('by key   :', dict(sorted(prices.items())))
print('cheap    :', {k: v for k, v in prices.items() if v < 15000})

## 5. Invert karna — duplicate values ko group karo

Simple inversion `{v: k for k, v in d.items()}` duplicate values par **data kho deta hai**. Grouping ke liye `defaultdict(list)` chahiye.


In [ ]:
marks = {'math': 80, 'science': 80, 'english': 70}
print('lossy  :', {v: k for k, v in marks.items()})

grouped = defaultdict(list)
for subject, score in marks.items():
    grouped[score].append(subject)
print('grouped:', {k: sorted(v) for k, v in sorted(grouped.items())})

## 6. Nested dictionaries — safe path aur flatten

Real data (JSON/config/API) nested hota hai. Do patterns zaroori hain: **deep get** (missing path par default) aur **flatten** (dot-path keys).


In [ ]:
def deep_get(data, path, default=None):
    current = data
    for part in path.split('.'):
        if not isinstance(current, dict) or part not in current:
            return default
        current = current[part]
    return current

config = {'db': {'host': 'localhost', 'ports': [5432]}, 'flags': {'rag': True}}
print(deep_get(config, 'db.host'), deep_get(config, 'db.port', 5432), deep_get(config, 'cache.ttl'))

def flatten(data, prefix='', out=None):
    out = {} if out is None else out         # mutable default argument se bacho
    for key, value in data.items():
        path = f'{prefix}{key}'
        if isinstance(value, dict):
            flatten(value, path + '.', out)
        else:
            out[path] = value
    return out

print(flatten({'a': 1, 'b': {'c': 2, 'd': {'e': 3}}}))
print(flatten(config))

## 7. Diff nikalna — do dictionaries ka comparison

`keys()` views set-like hain, isliye `-` aur `&` O(n+m) me chalta hai. Config drift detection me yahi pattern use hota hai.


In [ ]:
def diff(a, b):
    return {
        'added': {k: b[k] for k in b.keys() - a.keys()},
        'removed': {k: a[k] for k in a.keys() - b.keys()},
        'changed': {k: (a[k], b[k]) for k in a.keys() & b.keys() if a[k] != b[k]},
    }

print(diff({'x': 1, 'y': 2}, {'x': 1, 'y': 5, 'z': 3}))

def scrub(payload, allowed=('event', 'user_id', 'ts')):
    return {k: payload[k] for k in allowed if k in payload}

event = {'event': 'login', 'user_id': 42, 'ts': '2026-09-24', 'password': 'secret'}
print('scrubbed:', scrub(event))

## 8. Debugging — teen cells, teen alag problems

1. `KeyError` — direct lookup bina key.
2. `TypeError` — unhashable key (list) ko dictionary key banane ki koshish.
3. **Logic bug** — duplicate keys ka silent overwrite (koi error nahi aayega).


In [ ]:
user = {'name': 'Ali'}
print(user['city'])

In [ ]:
matrix = {}
matrix[[1, 2]] = 'value'

In [ ]:
counts = {'a': 1, 'b': 2, 'c': 3}
for key in counts:
    if counts[key] < 2:
        del counts[key]

In [ ]:
# Fix 1: get() ya explicit membership check
user = {'name': 'Ali'}
print(user.get('city', 'unknown'))

# Fix 2: tuple key (hashable) ya string key
matrix = {}
matrix[(1, 2)] = 'value'
print(matrix)

# Fix 3: iterate over a copy, ya comprehension se nayi dict banao
counts = {'a': 1, 'b': 2, 'c': 3}
for key in list(counts):
    if counts[key] < 2:
        del counts[key]
print(counts)
print({k: v for k, v in {'a': 1, 'b': 2}.items() if v >= 2})

### Duplicate keys ka silent data loss

Ye cell chalta rahega par output me `a` ki pehli value gayab hai — errors ke bina data khona sabse khatarnak bug hai.


In [ ]:
rows = [('a', 1), ('b', 2), ('a', 3)]

last_wins = {}
first_wins = {}
all_values = defaultdict(list)
for key, value in rows:
    last_wins[key] = value                 # last-write-wins
    first_wins.setdefault(key, value)      # first-wins
    all_values[key].append(value)          # kuch bhi na kho
print('last :', last_wins)
print('first:', first_wins)
print('all  :', dict(all_values))
print('duplicate keys:', sorted(k for k, n in Counter(k for k, _ in rows).items() if n > 1))

## 9. Practice set

1. `d1 = {'Mango':100,'Banana':200}` aur `d2 = {'Orange':300,'Kiwi':400}` merge karo (dono tarike).
2. Dictionary se keys hatao jinki value `None` hai (comprehension).
3. A → Z tak ASCII dictionary banao, phir even/odd values wali do dictionaries.
4. 1..10 ka square dictionary banao.
5. List of tuples (name, age, salary) ko salary par sort karo.
6. Ek nested dict se specific path ki value update karo (nayi dict banakar).
7. Cache dictionary me TTL check add karo.


In [ ]:
# 1
d1 = {'Mango': 100, 'Banana': 200}
d2 = {'Orange': 300, 'Kiwi': 400}
print({**d1, **d2})
merged = d1.copy()
merged.update(d2)
print(merged)

# 2
raw = {'a': 1, 'b': None, 'c': 3}
print({k: v for k, v in raw.items() if v is not None})

# 3
ascii_map = {chr(i): i for i in range(65, 91)}
print({k: v for k, v in list(ascii_map.items())[:4]})
print('even codes:', sum(1 for v in ascii_map.values() if v % 2 == 0))

# 4
print({n: n ** 2 for n in range(1, 11)})

# 5
people = [('Anjali', 28, 1320000), ('Salman', 29, 142000), ('Mona', 34, 123000)]
print(sorted(people, key=lambda p: p[2]))

# 6
def set_path(data, path, value):
    key, *rest = path.split('.')
    if not rest:
        return {**data, key: value}
    child = data.get(key) if isinstance(data.get(key), dict) else {}
    return {**data, key: set_path(child, '.'.join(rest), value)}

base = {'db': {'host': 'a'}, 'debug': False}
print(set_path(base, 'db.port', 5432))
print('original untouched:', base)

# 7
import time

cache = {}
def cached(key, compute, ttl=60):
    entry = cache.get(key)
    if entry and entry['expires'] > time.time():
        return entry['value']
    value = compute()
    cache[key] = {'value': value, 'expires': time.time() + ttl}
    return value

calls = {'n': 0}
def expensive():
    calls['n'] += 1
    return 'result'

cached('k', expensive)
cached('k', expensive)
print('compute called', calls['n'], 'time(s) | value:', cache['k']['value'])

## 10. Interview drills

1. Average lookup O(1) kyun hai? Worst case kya hai, aur Python usse kaise handle karta hai?
2. `d[key]` vs `d.get(key)` — kis situation me kaunsa, aur kyun?
3. Iteration ke dauraan dict ka size badalna kyun RuntimeError deta hai?
4. Bounded LRU cache dictionary se kaise banoge (O(1) get/put)?


In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.cap:
            self.data.popitem(last=False)   # sabse purani entry

cache = LRUCache(2)
cache.put(1, 'one')
cache.put(2, 'two')
print(cache.get(1))
cache.put(3, 'three')          # capacity 2 → 2 evict ho jayega
print(cache.get(2), cache.get(3), list(cache.data.items()))